# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/scour_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d07_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # scour train
    shutil.copy(glob.glob("/kaggle/input/**/d07_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # scour test
    print("Kaggle: scour data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: scour data copied to ./drive/MyDrive/Regression/scour_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 mapie==0.6.0 puncc==0.8.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from pytorch_tabnet import TabNetRegressor

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795363612.298530      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795363628.116014      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
# Go to find & replace button and replace (scour_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [5]:
feature_names = ['Ps', 'Pw', ' Skew  ', ' Velocity  ', ' Depth  ', ' D50  ', ' Gradation  ']
TARGET_LABEL = 'Scour depth'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [6]:
train_data_path = "./drive/MyDrive/Regression/scour_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/scour_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (154, 8)
First 5 rows of training data:
     Ps   Pw   Skew     Velocity     Depth     D50     Gradation     Scour  
0  0.7  1.5        0          2.9       6.6   70.00           1.2       0.6
1  0.7  1.5        0          3.0       5.3   70.00           1.2       0.6
2  1.3  3.3        0          0.8       4.8    0.48           1.8       0.4
3  1.0  4.3        0          2.9       9.7    0.30           1.4       3.7
4  0.7  0.5       20          1.4       1.8    1.10           3.5       0.3

Shape of test data: (78, 8)
First 5 rows of test data:
     Ps   Pw   Skew     Velocity     Depth     D50     Gradation     Scour  
0  1.3  0.3       16          1.0       0.3    0.94           3.0       0.4
1  1.0  0.8        0          0.2       1.5    0.25           8.0       0.2
2  1.0  0.6       10          1.6       6.6    0.90           4.2       1.1
3  1.3  1.2        0          0.8       3.1    0.38           2.3       1.6
4  1.0  0.8        0          1.1       1

In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (154, 7)
Shape of y_train: (154,)
Shape of X_test: (78, 7)
Shape of y_test: (78,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[-1.30505288 -0.05072804 -0.49867969  1.41932674  0.51129506  1.91265145
  -0.75772712]
 [-1.30505288 -0.05072804 -0.49867969  1.53187891  0.18675076  1.91265145
  -0.75772712]
 [ 1.56606345  1.51169548 -0.49867969 -0.94426887  0.06192603 -0.69389788
  -0.5721613 ]
 [ 0.13050529  2.37970854 -0.49867969  1.41932674  1.28520839 -0.70064672
  -0.69587185]
 [-1.30505288 -0.9187411   0.57841248 -0.26895584 -0.68702234 -0.6706519
  -0.04639146]]

First five rows of normalized X_test:
[[ 1.56606345 -1.09234371  0.36299405 -0.71916453 -1.06149653 -0.67665086
  -0.20102964]
 [ 0.13050529 -0.65833718 -0.49867969 -1.6195819  -0.76191718 -0.70252139
   1.34535224]
 [ 0.13050529 -0.83193979  0.0398664  -0.0438515   0.51129506 -0.6781506
   0.17010201]
 [ 1.56606345 -0.31113196 -0.49867969 -0.94426887 -0.36247805 -0.69764723
  -0.41752311]
 [ 0.13050529 -0.65833718 -0.49867969 -0.60661235 -0.71198729  1.53771627
  -0.38659547]]


# **Functions:**

In [10]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [11]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [12]:
# FIX Q2: quantile crossing. Each quantile level is a separate model, so for some rows the
# predictions can come out in the wrong order (e.g. q0.10 < q0.05). Before any metric we sort
# each row's quantile predictions (monotone rearrangement, Chernozhukov et al. 2010) and record
# how many rows crossed, for the Excel report.
quantile_crossing_report = {}

def rearrange_quantiles(predictions_df, tag=None):
    q_cols = sorted(c for c in predictions_df.columns if c != 'Actual')
    values = predictions_df[q_cols].to_numpy(dtype=float)
    crossed = (np.diff(values, axis=1) < 0).any(axis=1)
    if tag is not None:
        quantile_crossing_report[tag] = {'rows': int(len(values)), 'rows_crossing': int(crossed.sum()),
                                         'pct_crossing': round(100.0 * crossed.mean(), 2)}
    out = predictions_df.copy()
    out[q_cols] = np.sort(values, axis=1)
    return out


In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def quantile_regression_print(predictions_df, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Extract actual values
    actual_values = predictions_df['Actual'].values

    # Debugging: Print the first few actual values
    print("First few actual values:", actual_values[:5])

    # Calculate coverage for the specific interval (0.05 to 0.95)
    lower_quantile = 0.05
    median_quantile = 0.5
    upper_quantile = 0.95

    lower_preds = predictions_df[lower_quantile].values
    median_preds = predictions_df[median_quantile].values
    upper_preds = predictions_df[upper_quantile].values

    in_interval = ((actual_values >= lower_preds) & (actual_values <= upper_preds))
    coverage = in_interval.mean()
    print(f"Coverage of 90% prediction interval: {coverage * 100:.2f}%")

    # Create a new Excel workbook and add a worksheet
    wb = Workbook()
    ws = wb.active
    ws.title = "quantile_regression_print"

    # Plot actual vs. all predicted quantiles
    num_points = len(predictions_df)
    indices = np.arange(num_points)

    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values[:num_points], label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')

    # Loop over all columns assumed to be quantile predictions
    for column in predictions_df.columns:
        if column != 'Actual':  # Skip the actual values column
            plt.plot(indices, predictions_df[column][:num_points],
                     label=f'Predicted Quantile {column}', linestyle='--')

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title(f'Actual vs. Predicted Quantile {TARGET_LABEL}')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A1')

    # Plot actual values and specific prediction intervals (0.05, 0.5, 0.95)
    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values, label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')  # Connect points with a line
    plt.plot(indices, median_preds, label=f'Median Prediction ({median_quantile})', marker='x', linestyle='-', color='blue')
    plt.fill_between(
        indices,
        lower_preds,
        upper_preds,
        color='blue',
        alpha=0.5,
        label=f'Prediction Interval ({lower_quantile}-{upper_quantile})'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(f' {TARGET_LABEL}')
    plt.title(f'Actual {TARGET_LABEL} with Prediction Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1, 1))

    # Save the second plot to another BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the second image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A30')  # Adjust the cell position as needed

    # Save the workbook
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluation_features_print(predictions_df, X_test, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Ensure X_test is a DataFrame
    if isinstance(X_test, np.ndarray):
        X_test = pd.DataFrame(X_test, columns=feature_names)
    X_test.columns = feature_names

    # Get quantile columns from predictions_df except 'Actual'
    quantile_columns = [col for col in predictions_df.columns if col != 'Actual']

    # Check if the Excel file already exists
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="New Evaluation Features")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "evaluation_features_print"

    image_row = 1  # Where to start images

    for feature_to_plot in feature_names:
        feature_values = X_test[feature_to_plot]
        plot_df = predictions_df.copy()
        plot_df[feature_to_plot] = feature_values.values

        # Sort by current feature to plot lines well
        plot_df = plot_df.sort_values(by=feature_to_plot)

        # Begin Matplotlib plot
        plt.figure(figsize=(6, 4))
        # Plot each quantile as a line
        for quantile in quantile_columns:
            plt.plot(plot_df[feature_to_plot], plot_df[quantile], label=f"{quantile}")

        # Plot actual values as scatter
        plt.scatter(plot_df[feature_to_plot], plot_df['Actual'], color='black', s=12, label=f"Actual {TARGET_LABEL}")

        plt.title(f"Predicted Quantiles vs. {feature_to_plot}")
        plt.xlabel(feature_to_plot)
        plt.ylabel(f"{TARGET_LABEL}")
        plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
        plt.tight_layout()

        # Save figure to a BytesIO stream
        img_bytes = BytesIO()
        img_bytes
        plt.savefig(img_bytes, format="png")
        plt.close()
        img_bytes.seek(0)

        img = Image(img_bytes)
        img.anchor = f"A{image_row}"
        ws.add_image(img)

        image_row += 20  # Adjust as you like

    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluate_uncertainity_matrix(predictions_df, quantiles, excel_file_path, model_name='Model'):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2

   # Extract the actual target values
    y_true = predictions_df['Actual']

    # Open existing Excel file
    wb = load_workbook(_ensure_excel_file(excel_file_path))

    # Create unique sheet names
    def get_unique_sheet_name(wb, base_name):
        sheet_name = base_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base_name}_{i}"
            i += 1
        return sheet_name

    # Create separate sheets for plots and values
    plots_sheet_title = get_unique_sheet_name(wb, 'Plots')
    values_sheet_title = get_unique_sheet_name(wb, 'Values')

    ws_plots = wb.create_sheet(title=plots_sheet_title)
    ws_values = wb.create_sheet(title=values_sheet_title)
    # FIX Q2: how often the raw quantile predictions crossed (before sorting)
    _cr = quantile_crossing_report.get(excel_file_path, {})
    ws_values.append(['Rows with quantile crossing (before sorting)', _cr.get('rows_crossing'), 'of', _cr.get('rows'), 'rows', f"{_cr.get('pct_crossing')}%"])
    print(f"Quantile crossing before sorting: {_cr.get('rows_crossing')} of {_cr.get('rows')} rows ({_cr.get('pct_crossing')}%)")

    # Positions to place images
    image_positions = ['A1', 'A25', 'A49', 'A73']  # Adjust as needed

    ### 1. Validity / Coverage ###

    # Compute empirical coverage for each quantile
    empirical_quantiles = []
    for q in quantiles:
        coverage = (predictions_df[q] >= y_true).mean()
        empirical_quantiles.append(coverage)

    # Plot theoretical quantiles vs empirical quantiles
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=quantiles, y=quantiles, color="magenta", linestyle='--', linewidth=2, label="Ideal")
    sns.lineplot(x=quantiles, y=empirical_quantiles, color="blue", linewidth=2, label="Empirical")
    sns.scatterplot(x=quantiles, y=empirical_quantiles, color="blue", s=50)
    plt.xlabel("Nominal Quantile Levels")
    plt.ylabel("Empirical Quantile Levels")
    plt.title(f"Validity / Coverage - {model_name}")
    plt.legend()

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add the image to the plots worksheet
    ws_plots.add_image(img, image_positions[0])

    ### 2. Sharpness / Interval Length ###

    # Define coverage levels to evaluate
    coverage_levels = [0.6, 0.8, 0.9]
    average_interval_lengths = []

    # Compute average interval lengths for each coverage level
    for c in coverage_levels:
        lower_q = (1 - c) / 2
        upper_q = 1 - lower_q
        # Find the nearest quantiles in our quantiles list
        lower_quantile = min(quantiles, key=lambda x: abs(x - lower_q))
        upper_quantile = min(quantiles, key=lambda x: abs(x - upper_q))

        # Compute interval length for each observation
        interval_length = predictions_df[upper_quantile] - predictions_df[lower_quantile]
        avg_interval_length = interval_length.mean()
        average_interval_lengths.append(avg_interval_length)
        # Write interval length information to the values worksheet
        ws_values.append([f'Coverage level: {c*100:.0f}%', f'Interval: [{lower_quantile}, {upper_quantile}]', f'Average Interval Length: {avg_interval_length:.4f}'])

    # Plot coverage levels vs average interval lengths
    plt.figure(figsize=(10, 6))
    plt.plot([c*100 for c in coverage_levels], average_interval_lengths, marker='o')
    plt.xlabel('Coverage Level (%)')
    plt.ylabel('Average Interval Length')
    plt.title(f'Sharpness / Interval Length - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[1])

    ### 3. Negative Log-Likelihood (NLL) ###

    # Use the 5th and 95th percentiles to estimate standard deviation
    if 0.05 in quantiles and 0.95 in quantiles:
        z_lower = norm.ppf(0.05)  # z-score for 5% quantile (~ -1.6449)
        z_upper = norm.ppf(0.95)  # z-score for 95% quantile (~ +1.6449)

        # Extract quantile predictions
        q_lower = predictions_df[0.05]
        q_upper = predictions_df[0.95]

    elif 0.1 in quantiles and 0.9 in quantiles:
        z_lower = norm.ppf(0.1)  # z-score for 10% quantile (~ -1.2816)
        z_upper = norm.ppf(0.9)  # z-score for 90% quantile (~ +1.2816)

        # Extract quantile predictions
        q_lower = predictions_df[0.1]
        q_upper = predictions_df[0.9]
    else:
        raise ValueError("Required quantiles for NLL estimation not found in quantiles list.")

    # Use median as the mean estimate
    if 0.5 in quantiles:
        q_median = predictions_df[0.5]
    else:
        # If 0.5 quantile is not available, use the middle quantile
        q_median = predictions_df[quantiles[len(quantiles)/2]]

    # Estimate standard deviation for each observation
    std_estimates = (q_upper - q_lower) / (z_upper - z_lower)

    # Ensure standard deviations are positive and non-zero
    std_estimates = std_estimates.clip(lower=1e-6)

    # Extract mean estimates (median predictions)
    mean_estimates = q_median

    # Compute Negative Log-Likelihood for each observation
    nll = -norm.logpdf(y_true, loc=mean_estimates, scale=std_estimates)

    # Compute average NLL
    average_nll = nll.mean()
    # Write average NLL to the values worksheet
    ws_values.append(['Average Negative Log-Likelihood (NLL)', average_nll])

    # Write NLL values to the values worksheet
    ws_values.append(['NLL Values'])
    for value in nll:
        ws_values.append([value])

    # Plot ECDF of NLL values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(nll, color='blue', linewidth=2)
    plt.xlabel('Negative Log-Likelihood (NLL)')
    plt.ylabel('ECDF')
    plt.title(f'NLL ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[2])

    ### 4. Continuous Ranked Probability Score (CRPS) ###

    # Prepare ensemble of quantile predictions for each observation
    ensemble_predictions = predictions_df[quantiles].to_numpy()

    # Ensure quantile predictions are sorted for each observation
    ensemble_predictions.sort(axis=1)

    # Compute CRPS for each observation using ensemble predictions
    # FIX Q4: CRPS from a set of quantiles = 2 * integral of the pinball loss over tau.
    # Each quantile level gets a weight equal to the share of [0, 1] it represents
    # (levels are unevenly spaced). Was: ps.crps_ensemble, which treats the 13 quantiles as
    # 13 equally likely samples.
    _taus = np.array(quantiles, dtype=float)
    _edges = np.concatenate([[0.0], (_taus[1:] + _taus[:-1]) / 2.0, [1.0]])
    _w = np.diff(_edges)
    _err = np.asarray(y_true, dtype=float)[:, None] - ensemble_predictions
    _pinball = np.maximum(_taus * _err, (_taus - 1.0) * _err)
    crps_values = 2.0 * (_pinball * _w).sum(axis=1)

    # Compute average CRPS
    average_crps = crps_values.mean()
    # Write average CRPS to the values worksheet
    ws_values.append(['Average Continuous Ranked Probability Score (CRPS)', average_crps])

    # Write CRPS values to the values worksheet
    ws_values.append(['CRPS Values'])
    for value in crps_values:
        ws_values.append([value])

    # Plot ECDF of CRPS values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(crps_values, color='green', linewidth=2)
    plt.xlabel('Continuous Ranked Probability Score (CRPS)')
    plt.ylabel('ECDF')
    plt.title(f'CRPS ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[3])

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    wb.save(_ensure_parent_dir(excel_file_path))

    # Return a dictionary of results
    results = {
        'empirical_quantiles': empirical_quantiles,
        'average_interval_lengths': average_interval_lengths,
        'coverage_levels': coverage_levels,
        'average_nll': average_nll,
        'average_crps': average_crps,
        'nll_values': nll,
        'crps_values': crps_values
    }

    return results

In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def calibrate_and_plot_intervals(predictions_df, excel_file_path, alpha=0.1, max_samples=1000, random_seed=42):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2
    # Extract necessary columns
    lower = predictions_df[0.05].fillna(0)
    upper = predictions_df[0.95].fillna(0)
    pred = predictions_df[0.5].fillna(0)
    labels = predictions_df['Actual'].fillna(0)

    # Problem setup
    total_samples = labels.shape[0]
    n = min(max_samples, total_samples)  # Ensure n does not exceed the number of samples

    # Ensure there are enough samples for both calibration and validation
    if total_samples <= n:
        n = total_samples // 2

    # Split the data into calibration and validation sets
    np.random.seed(random_seed)  # Set the random seed for reproducibility
    idx = np.array([1] * n + [0] * (total_samples - n)) > 0
    np.random.shuffle(idx)
    cal_labels, val_labels = labels[idx], labels[~idx]
    cal_upper, val_upper = upper[idx], upper[~idx]
    cal_lower, val_lower = lower[idx], lower[~idx]
    cal_pred, val_pred = pred[idx], pred[~idx]

    # Calculate uncertainty intervals
    cal_U = cal_upper - cal_lower
    val_U = val_upper - val_lower

    # Avoid division by zero by replacing zero intervals with a small number
    cal_U[cal_U == 0] = np.finfo(float).eps
    val_U[val_U == 0] = np.finfo(float).eps

    # Get scores
    cal_scores = np.abs(cal_pred - cal_labels) / cal_U

    # Debugging: Check for NaN in scores
    if np.isnan(cal_scores).any():
        print("NaN values found in cal_scores. Check data integrity.")

    # Get the score quantile
    qhat = np.quantile(cal_scores, np.ceil((n + 1) * (1 - alpha)) / n, interpolation='higher')

    # Deploy (output=lower and upper adjusted quantiles)
    prediction_sets = [val_pred - val_U * qhat, val_pred + val_U * qhat]

    # Calculate empirical coverage (before and after calibration)
    prediction_sets_uncalibrated = [val_lower, val_upper]
    empirical_coverage_uncalibrated = ((val_labels >= prediction_sets_uncalibrated[0]) & (val_labels <= prediction_sets_uncalibrated[1])).mean() * 100
    print(f"The empirical coverage before calibration is: {empirical_coverage_uncalibrated:.2f}%")

    empirical_coverage = ((val_labels >= prediction_sets[0]) & (val_labels <= prediction_sets[1])).mean() * 100
    print(f"The empirical coverage after calibration is: {empirical_coverage:.2f}%")

    # Create a DataFrame with all the necessary data
    data = pd.DataFrame({
        'Index': np.arange(len(val_labels)),
        'Uncalibrated Lower': val_lower,
        'Uncalibrated Upper': val_upper,
        'Calibrated Lower': prediction_sets[0],
        'Calibrated Upper': prediction_sets[1],
        'True Label': val_labels,
        'Predicted Value': val_pred
    })

    plt.figure(figsize=(14, 8))

    # Plot the uncalibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Uncalibrated Lower'],
        data['Uncalibrated Upper'],
        color='blue',
        alpha=0.5,
        label='Uncalibrated Prediction Interval'
    )

    # Plot the calibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Calibrated Lower'],
        data['Calibrated Upper'],
        color='lightgreen',
        alpha=0.5,
        label='Calibrated Prediction Interval'
    )

    # Plot the true labels as points and connect them with a line
    plt.plot(
        data['Index'],
        data['True Label'],
        'o-',
        color='red',
        markersize=4,
        label='Actual Value'
    )

    # Plot the predicted values as points and connect them with a line
    plt.plot(
        data['Index'],
        data['Predicted Value'],
        'o-',
        color='black',
        markersize=4,
        label='Predicted Value'
    )

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title('Calibrated and Uncalibrated Prediction Intervals with True and Predicted Values')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()

    # Save the plot as an image
    image_path = 'plot.png'
    _ensure_parent_dir(image_path)
    plt.savefig(_ensure_parent_dir(image_path))
    plt.close()

    # Load the existing Excel file
    workbook = load_workbook(_ensure_excel_file(excel_file_path))

    # Create a new sheet for the plot
    new_sheet_name = "calibrate_and_plot_intervals"
    workbook.create_sheet(title=new_sheet_name)

    # Insert the image into the new sheet
    sheet = workbook[new_sheet_name]
    img = Image(image_path)
    sheet.add_image(img, 'A1')

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    workbook.save(_ensure_parent_dir(excel_file_path))

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def plot_quantile_intervals_to_excel(predictions_df, file_path, alpha=0.1):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    """
    Plots conformal prediction intervals and their coverage, and saves the plots to an Excel file.

    Parameters:
    - predictions_df: DataFrame containing prediction quantiles and actual values.
    - file_path: Path to the Excel file where plots will be saved.
    - alpha: Desired coverage level (default is 0.1).
    """
    # Extract mean prediction and uncertainty
    mean_prediction = predictions_df[0.5]
    uncertainty = predictions_df[0.95] - predictions_df[0.05]
    actual_values = predictions_df['Actual']

    # Calculate conformal scores
    conformal_scores = np.abs(mean_prediction - actual_values) / uncertainty

    # Function to find weighted quantile
    def weighted_quantile(values, quantile, sample_weight=None):
        values = np.array(values)
        if sample_weight is None:
            sample_weight = np.ones(len(values))
        sorter = np.argsort(values)
        values, sample_weight = values[sorter], sample_weight[sorter]
        weighted_quantiles = np.cumsum(sample_weight) - 0.5 * sample_weight
        weighted_quantiles /= np.sum(sample_weight)
        return np.interp(quantile, weighted_quantiles, values)

    # Calculate weighted quantile
    weights = np.ones_like(conformal_scores)  # Uniform weights for simplicity
    quantile = weighted_quantile(conformal_scores, 1 - alpha, sample_weight=weights)

    # Calculate prediction intervals
    lower_bound = mean_prediction - quantile * uncertainty
    upper_bound = mean_prediction + quantile * uncertainty

    # Naive conformal prediction
    naive_quantile = np.quantile(conformal_scores, 1 - alpha)
    naive_lower_bound = mean_prediction - naive_quantile * uncertainty
    naive_upper_bound = mean_prediction + naive_quantile * uncertainty

    # Coverage calculation (not plotted, just for potential diagnostics)
    weighted_coverage_points = (actual_values >= lower_bound) & (actual_values <= upper_bound)
    naive_coverage_points = (actual_values >= naive_lower_bound) & (actual_values <= naive_upper_bound)

    # Visualization using Matplotlib
    fig, ax = plt.subplots(figsize=(10, 5))

    x = np.arange(len(predictions_df))
    # Actual Values
    ax.plot(x, actual_values, label='Actual', color='black', linewidth=2)

    # Mean Prediction
    ax.plot(x, mean_prediction, label='Mean Prediction', color='darkorange', linewidth=2)

    # Weighted Prediction Interval (blue fill)
    ax.fill_between(x, lower_bound, upper_bound, color='blue', alpha=0.3, label='Weighted Interval')
    # Optionally, plot the bounds as lines if you want:
    # ax.plot(x, lower_bound, color='blue', linewidth=1, linestyle='--')
    # ax.plot(x, upper_bound, color='blue', linewidth=1, linestyle='--')

    # Naive Prediction Interval (red fill)
    ax.fill_between(x, naive_lower_bound, naive_upper_bound, color='red', alpha=0.3, label='Naive Interval')
    # Optionally, plot the bounds as lines:
    # ax.plot(x, naive_lower_bound, color='red', linewidth=1, linestyle='--')
    # ax.plot(x, naive_upper_bound, color='red', linewidth=1, linestyle='--')

    ax.set_title('Quantile Prediction Intervals')
    ax.set_xlabel('Sample Number')
    ax.set_ylabel(f'{TARGET_LABEL}')
    ax.grid(False)
    ax.set_facecolor('white')

    # Place the legend outside the plot
    ax.legend(loc='center left', bbox_to_anchor=(1.05, 0.5), borderaxespad=0.0)

    plt.tight_layout(rect=[0,0,0.80,1.0])  # Make space for legend

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Add image to Excel
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="plot_quantile_intervals_to_excel")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "plot_quantile_intervals_to_excel"

    img = Image(img_data)
    img.anchor = 'A1'
    ws.add_image(img)
    ws.column_dimensions['A'].width = 30
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

# **Tuned Parameters**

In [18]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/scour_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshjangra/scour-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


In [19]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 0.3167492200754044,
  'cv_rmse': 0.5628047797197572,
  'best_params': {'n_estimators': 500,
   'criterion': 'poisson',
   'max_depth': None,
   'min_samples_split': 2,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 1.0,
   'max_leaf_nodes': None,
   'min_impurity_decrease': 0.0,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.001},
  'test_mse': 0.2086811466977145,
  'test_rmse': np.float64(0.4568163161465607),
  'test_corr_coef': np.float64(0.9027344030495817),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 0.31674922007540446,
  'cv_rmse': 0.5628047797197572,
  'best_params': {'n_estimators': 500,
   'criterion': 'poisson',
   'max_depth': None,
   'min_samples_split': 2,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 1.0,
   'max_leaf_nodes': None,
   'min_impurity_decrease': 0.0,
   'n

# **Quantile Regression with Lightgbm**

In [20]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_LGBM_df = pd.DataFrame()


# Train a model for each quantile and make predictions using LightGBM
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = LGBMRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_LGBM_df[q] = predictions

# Add actual target values to the DataFrame
predictions_LGBM_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_LGBM_df.head())

   0.05       0.1      0.15       0.2       0.3       0.4       0.5       0.6  \
0   0.2  0.300000  0.290133  0.296233  0.363646  0.396124  0.410967  0.377134   
1   0.2  0.300000  0.282512  0.189664  0.163723  0.132062  0.130339  0.129705   
2   0.2  0.727616  0.656817  0.725924  0.692872  0.871365  0.931937  0.994027   
3   0.3  0.292311  0.231266  0.350918  0.367861  0.427837  0.486532  0.533371   
4   0.2  0.200000  0.257321  0.302246  0.297824  0.293460  0.501477  0.491865   

        0.7       0.8      0.85       0.9      0.95  Actual  
0  0.442071  0.475144  0.463033  0.439554  0.477381     0.4  
1  0.186551  0.346462  0.370413  0.355268  0.578697     0.2  
2  1.012137  1.067660  1.149876  1.147053  1.245063     1.1  
3  0.622974  0.931775  1.024733  0.975790  0.956138     1.6  
4  0.519952  0.592529  0.521048  0.648893  0.724120     0.2  


In [21]:
quantile_regression_print(predictions_LGBM_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

First few actual values: [0.4 0.2 1.1 1.6 0.2]
Coverage of 90% prediction interval: 76.92%


In [22]:
evaluation_features_print(predictions_LGBM_df,x_test,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

In [23]:
evaluate_uncertainity_matrix(predictions_LGBM_df, quantiles, "./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/LightGBM.xlsx", model_name='LightGBM')

Quantile crossing before sorting: 78 of 78 rows (100.0%)


{'empirical_quantiles': [np.float64(0.10256410256410256),
  np.float64(0.1794871794871795),
  np.float64(0.19230769230769232),
  np.float64(0.23076923076923078),
  np.float64(0.32051282051282054),
  np.float64(0.3717948717948718),
  np.float64(0.46153846153846156),
  np.float64(0.5512820512820513),
  np.float64(0.6282051282051282),
  np.float64(0.6794871794871795),
  np.float64(0.7692307692307693),
  np.float64(0.7948717948717948),
  np.float64(0.8717948717948718)],
 'average_interval_lengths': [np.float64(0.39814773667909914),
  np.float64(0.688546153615537),
  np.float64(1.1564932039391482)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(0.7567296266304484),
 'average_crps': np.float64(0.21162641346726596),
 'nll_values': array([-1.55316520e+00, -1.07261019e+00, -8.78416943e-02,  1.01525206e+01,
         7.60079540e-01,  2.80629227e+00,  2.80176210e-01, -2.78867467e-01,
        -6.28285757e-01,  8.89639620e-01, -9.83621488e-01, -1.82412594e-01,
         1.22104212e+

In [24]:
calibrate_and_plot_intervals(predictions_LGBM_df, "./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

The empirical coverage before calibration is: 74.36%
The empirical coverage after calibration is: 92.31%


In [25]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_LGBM_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

# **Quantile Regression with XGBoost**

In [26]:
# Get the best hyperparameters for XGBRegressor
best_paramst = get_best_model_params(best_scores_autosampler, 'XGBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_XGB_df = pd.DataFrame()

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_paramst.copy()  # FIX Q1: was best_params (= LightGBM's settings from the cell above)

    # Adjust according to your XGBoost version
    params.update({
        'objective': 'reg:quantileerror',  # Use the correct objective
        'quantile_alpha': q,                         # Use 'alpha' to set the quantile level
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = XGBRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_XGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_XGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_XGB_df.head())

       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0  0.271528  0.310806  0.280158  0.288421  0.262889  0.352578  0.368271   
1  0.171888  0.171551  0.202176  0.339029  0.210683  0.179865  0.247269   
2  0.779851  0.825209  0.876234  0.765110  1.223332  0.745544  0.977628   
3  0.356153  0.383716  0.456246  0.482912  0.515239  0.982884  0.559371   
4  0.138087  0.254094  0.247723  0.197444  0.170978  0.373044  0.600618   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  0.355777  0.357256  0.387720  0.412067  0.628268  0.620713     0.4  
1  0.258819  0.419875  0.581645  0.349233  0.538381  0.834920     0.2  
2  0.988500  0.623149  1.311807  0.958821  1.116188  0.930736     1.1  
3  1.037321  1.114164  1.404182  0.767300  1.089287  1.371320     1.6  
4  0.579804  0.454390  0.585792  0.618240  0.531743  0.576493     0.2  


In [27]:
quantile_regression_print(predictions_XGB_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

First few actual values: [0.4 0.2 1.1 1.6 0.2]
Coverage of 90% prediction interval: 76.92%


In [28]:
evaluation_features_print(predictions_XGB_df,x_test,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

In [29]:
evaluate_uncertainity_matrix(predictions_XGB_df, quantiles, "./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/XGBoost.xlsx", model_name='XGBoost')

Quantile crossing before sorting: 77 of 78 rows (98.72%)


{'empirical_quantiles': [np.float64(0.1282051282051282),
  np.float64(0.1282051282051282),
  np.float64(0.16666666666666666),
  np.float64(0.28205128205128205),
  np.float64(0.38461538461538464),
  np.float64(0.46153846153846156),
  np.float64(0.5),
  np.float64(0.5128205128205128),
  np.float64(0.5512820512820513),
  np.float64(0.6153846153846154),
  np.float64(0.7051282051282052),
  np.float64(0.7692307692307693),
  np.float64(0.8974358974358975)],
 'average_interval_lengths': [np.float64(0.32462902214282596),
  np.float64(0.5800530056540782),
  np.float64(0.8611099953070666)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(0.6686015335216425),
 'average_crps': np.float64(0.20222173891866055),
 'nll_values': array([-1.19941211, -0.63974252, -0.31797629,  3.1910276 ,  0.51338025,
         3.74559632,  0.36841343, -0.43738622, -1.07310549,  0.48310919,
        -0.91312976, -0.76236135,  0.71769802, -0.46103614, -0.81449868,
        -0.35152765, -0.42129634,  0.51092677

In [30]:
calibrate_and_plot_intervals(predictions_XGB_df, "./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

The empirical coverage before calibration is: 66.67%
The empirical coverage after calibration is: 87.18%


In [31]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_XGB_df, "./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

# **Quantile Regression with GPBoost**

In [32]:
# Get the best hyperparameters for GPBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GPBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'alpha', 'quantile_alpha', 'boosting_type', 'metric', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = GPBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GPBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GPBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GPBoost_df.head())

       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0  0.239854  0.300000  0.300000  0.292341  0.357027  0.416300  0.460366   
1  0.172290  0.300000  0.300000  0.225851  0.162707  0.170041  0.179395   
2  0.423603  0.722197  0.637593  0.776813  0.692579  0.821316  0.984733   
3  0.295176  0.300967  0.171251  0.374810  0.393466  0.444996  0.489650   
4  0.189765  0.200000  0.200000  0.319120  0.280927  0.356649  0.510425   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  0.409654  0.418585  0.483294  0.492805  0.494392  0.601201     0.4  
1  0.128631  0.241043  0.375085  0.587582  0.654390  1.290241     0.2  
2  1.011589  1.004013  1.089537  1.126418  1.243530  1.202344     1.1  
3  0.648401  0.715344  1.006400  1.008628  0.997278  0.920643     1.6  
4  0.442597  0.522702  0.553125  0.577100  0.723270  0.922043     0.2  


In [33]:
quantile_regression_print(predictions_GPBoost_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

First few actual values: [0.4 0.2 1.1 1.6 0.2]
Coverage of 90% prediction interval: 87.18%


In [34]:
evaluation_features_print(predictions_GPBoost_df,x_test,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

In [35]:
evaluate_uncertainity_matrix(predictions_GPBoost_df, quantiles,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/GPBoost.xlsx", model_name='GPBoost')

Quantile crossing before sorting: 76 of 78 rows (97.44%)


{'empirical_quantiles': [np.float64(0.08974358974358974),
  np.float64(0.16666666666666666),
  np.float64(0.19230769230769232),
  np.float64(0.21794871794871795),
  np.float64(0.3333333333333333),
  np.float64(0.4358974358974359),
  np.float64(0.5),
  np.float64(0.5769230769230769),
  np.float64(0.6410256410256411),
  np.float64(0.7307692307692307),
  np.float64(0.7948717948717948),
  np.float64(0.8461538461538461),
  np.float64(0.9615384615384616)],
 'average_interval_lengths': [np.float64(0.44850509638161856),
  np.float64(0.8050231047274026),
  np.float64(1.2162691129224041)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(0.43442811598709025),
 'average_crps': np.float64(0.20714544981049565),
 'nll_values': array([-1.27876506, -0.1152975 , -0.36345858,  9.06463246,  0.01043315,
         1.99655054,  0.17852858, -0.30856443, -0.32166007,  1.07107048,
        -0.99225004, -0.42193246,  1.24794424, -0.29780522, -0.63659625,
        -0.39421351,  0.03283792,  0.0794095

In [36]:
calibrate_and_plot_intervals(predictions_GPBoost_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

The empirical coverage before calibration is: 84.62%
The empirical coverage after calibration is: 94.87%


In [37]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GPBoost_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

# **Quantile Regression with NGBoost**

In [38]:
# Correct best_params assignment
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

# DataFrame to store predictions
predictions_NGB_df = pd.DataFrame()

# Remove 'Score' and 'Dist' if present in best_params
for key in ['Score', 'Dist']:
    best_params.pop(key, None)

# Update params with model settings
best_params.update({
    'Dist': Normal,
    'Score': LogScore,
    'random_state': 42,
    'verbose': True
})

# Initialize and fit the NGBoost model once
model = NGBRegressor(**best_params)

# Fit the model
model.fit(X_train, y_train)

# Make predictions
pred_dist = model.pred_dist(X_test)

# Extract the mean (mu) and standard deviation (sigma) of the predicted distributions
mu = pred_dist.loc    # Mean predictions
sigma = pred_dist.scale  # Standard deviation predictions

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# For each quantile, compute predictions
for q in quantiles:
    # Calculate the z-score for the desired quantile
    z = norm.ppf(q)

    # Compute the quantile prediction per sample
    predictions = mu + z * sigma

    # Add predictions to the DataFrame
    predictions_NGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_NGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_NGB_df.head())

[iter 0] loss=1.4735 val_loss=0.0000 scale=2.0000 norm=1.8577


[iter 100] loss=-0.0700 val_loss=0.0000 scale=2.0000 norm=0.8014


       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0  0.186238  0.224843  0.250890  0.271591  0.305299  0.334101  0.361023   
1 -0.130103 -0.068918 -0.027637  0.005173  0.058597  0.104247  0.146914   
2  0.586311  0.648614  0.690649  0.724057  0.778458  0.824940  0.868387   
3  0.365527  0.445451  0.499374  0.542231  0.612017  0.671647  0.727381   
4  0.127835  0.184963  0.223507  0.254141  0.304023  0.346645  0.386484   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  0.387944  0.416746  0.450454  0.471155  0.497202  0.535807     0.4  
1  0.189582  0.235231  0.288656  0.321465  0.362746  0.423932     0.2  
2  0.911834  0.958317  1.012717  1.046125  1.088160  1.150463     1.1  
3  0.783115  0.842744  0.912530  0.955387  1.009311  1.089234     1.6  
4  0.426322  0.468944  0.518826  0.549460  0.588004  0.645132     0.2  


In [39]:
quantile_regression_print(predictions_NGB_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

First few actual values: [0.4 0.2 1.1 1.6 0.2]
Coverage of 90% prediction interval: 66.67%


In [40]:
evaluation_features_print(predictions_NGB_df,x_test,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

In [41]:
evaluate_uncertainity_matrix(predictions_NGB_df, quantiles,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/NGBoost.xlsx", model_name='NGBoost')

Quantile crossing before sorting: 0 of 78 rows (0.0%)


{'empirical_quantiles': [np.float64(0.14102564102564102),
  np.float64(0.21794871794871795),
  np.float64(0.28205128205128205),
  np.float64(0.2948717948717949),
  np.float64(0.3974358974358974),
  np.float64(0.44871794871794873),
  np.float64(0.48717948717948717),
  np.float64(0.6025641025641025),
  np.float64(0.6923076923076923),
  np.float64(0.7435897435897436),
  np.float64(0.7435897435897436),
  np.float64(0.7692307692307693),
  np.float64(0.8076923076923077)],
 'average_interval_lengths': [np.float64(0.30061262431449),
  np.float64(0.45774816977610044),
  np.float64(0.5875134154017887)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(1.4677352358094302),
 'average_crps': np.float64(0.2141740462103197),
 'nll_values': array([-1.25564096, -0.81270864,  0.06775666,  7.27176672, -0.22778749,
        24.15854197,  0.32284338, -0.32980587, -0.98157886,  2.6790816 ,
        -0.98802851, -0.6202062 ,  0.67985131, -0.60511518, -1.09933605,
        -0.31340503,  0.2946463 

In [42]:
calibrate_and_plot_intervals(predictions_NGB_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

The empirical coverage before calibration is: 61.54%
The empirical coverage after calibration is: 92.31%


In [43]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_NGB_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

# **Quantile Regression with Gradient Boosting**

In [44]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')


# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GBR_df = pd.DataFrame()

# Custom quantile loss function
def quantile_loss(q, y_true, y_pred):
    e = y_true - y_pred
    return np.mean(np.maximum(q * e, (q - 1) * e))

# Train a model for each quantile and make predictions using GradientBoostingRegressor
for q in quantiles:

    params = best_params.copy()

    params.update({
        'loss': 'quantile',
        'alpha': q,
        'random_state': 42
    })
    # Initialize the model with quantile-specific settings
    model = GradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GBR_df.head())

       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0  0.259197  0.280186  0.306040  0.288856  0.321957  0.233907  0.400700   
1  0.171109  0.168624  0.148649  0.166228  0.153093  0.323254  0.366817   
2  0.486083  0.598373  0.714133  0.584321  0.936502  0.902868  1.038594   
3  0.314477  0.292247  0.257528  0.305028  0.386718  0.650704  0.678783   
4  0.209000  0.282648  0.189795  0.320420  0.347831  0.372544  0.421154   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  0.382717  0.289653  0.406662  0.602919  0.519280  0.723410     0.4  
1  0.398100  0.309239  0.675631  0.773390  0.790251  1.261326     0.2  
2  1.096309  1.214627  0.825527  1.023332  1.079499  1.007356     1.1  
3  0.736855  0.768976  0.957796  1.171750  1.163861  1.135610     1.6  
4  0.568872  0.563729  0.510701  0.661454  0.643574  1.030720     0.2  


In [45]:
quantile_regression_print(predictions_GBR_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

First few actual values: [0.4 0.2 1.1 1.6 0.2]
Coverage of 90% prediction interval: 84.62%


In [46]:
evaluation_features_print(predictions_GBR_df,x_test,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

In [47]:
evaluate_uncertainity_matrix(predictions_GBR_df, quantiles,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx", model_name='GBM')

Quantile crossing before sorting: 77 of 78 rows (98.72%)


{'empirical_quantiles': [np.float64(0.10256410256410256),
  np.float64(0.15384615384615385),
  np.float64(0.1794871794871795),
  np.float64(0.24358974358974358),
  np.float64(0.32051282051282054),
  np.float64(0.3974358974358974),
  np.float64(0.46153846153846156),
  np.float64(0.5256410256410257),
  np.float64(0.5897435897435898),
  np.float64(0.6666666666666666),
  np.float64(0.717948717948718),
  np.float64(0.8333333333333334),
  np.float64(0.9487179487179487)],
 'average_interval_lengths': [np.float64(0.4737795563275938),
  np.float64(0.8755538838281702),
  np.float64(1.2471600177195794)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(0.46184159171265404),
 'average_crps': np.float64(0.2101464961760143),
 'nll_values': array([-8.48679412e-01, -9.86943352e-02, -3.16049072e-01,  5.13265628e+00,
        -7.08637881e-02,  1.48515656e+00,  5.65361697e-01, -3.49048845e-01,
        -3.37821056e-01,  1.12776342e+00, -7.96246165e-01, -2.00683407e-01,
         1.29987085e+0

In [48]:
calibrate_and_plot_intervals(predictions_GBR_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

The empirical coverage before calibration is: 79.49%
The empirical coverage after calibration is: 87.18%


In [49]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GBR_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

# **Quantile Regression with CatBoost**

In [50]:
# Get the best hyperparameters for CatBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_CatBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'loss_function', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'loss_function': 'Quantile:alpha={}'.format(q),
        'random_seed': 42
    })

    # Initialize the model with the updated parameters
    model = CatBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train, verbose=0)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_CatBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_CatBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_CatBoost_df.head())

       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0  0.268937  0.289698  0.284481  0.291400  0.342080  0.342639  0.354773   
1  0.252266  0.251770  0.214689  0.247217  0.234826  0.254298  0.258783   
2  0.537047  0.577822  0.588791  0.670721  0.752501  0.813082  0.902268   
3  0.316187  0.364150  0.416715  0.446482  0.445222  0.514058  0.548977   
4  0.198451  0.217476  0.269779  0.317974  0.376700  0.409877  0.470844   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  0.388171  0.415866  0.462282  0.474128  0.538413  0.673404     0.4  
1  0.337127  0.380946  0.601504  0.648916  0.596492  0.529669     0.2  
2  1.049674  1.119439  1.059977  1.162178  1.151289  1.329976     1.1  
3  0.612672  0.716701  0.895944  1.033075  1.253942  1.470755     1.6  
4  0.521069  0.598707  0.657598  0.724795  0.750141  0.934092     0.2  


In [51]:
quantile_regression_print(predictions_CatBoost_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

First few actual values: [0.4 0.2 1.1 1.6 0.2]
Coverage of 90% prediction interval: 70.51%


In [52]:
evaluation_features_print(predictions_CatBoost_df,x_test,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

In [53]:
evaluate_uncertainity_matrix(predictions_CatBoost_df, quantiles,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/CatBoost.xlsx", model_name='CatBoost')

Quantile crossing before sorting: 47 of 78 rows (60.26%)


{'empirical_quantiles': [np.float64(0.19230769230769232),
  np.float64(0.20512820512820512),
  np.float64(0.20512820512820512),
  np.float64(0.2564102564102564),
  np.float64(0.28205128205128205),
  np.float64(0.4230769230769231),
  np.float64(0.48717948717948717),
  np.float64(0.5256410256410257),
  np.float64(0.6282051282051282),
  np.float64(0.7435897435897436),
  np.float64(0.782051282051282),
  np.float64(0.8846153846153846),
  np.float64(0.8974358974358975)],
 'average_interval_lengths': [np.float64(0.4452390526433996),
  np.float64(0.8124346070573131),
  np.float64(1.0190444362535593)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(0.5415732077754304),
 'average_crps': np.float64(0.20013035979713661),
 'nll_values': array([-1.10938827, -1.00688362, -0.16739459,  4.35591038,  0.15461154,
         2.06028331,  0.17076677, -0.29742253, -0.49928269,  0.47000354,
        -1.07405363, -0.7097422 ,  0.87391917, -0.34912737, -0.70057715,
        -0.20936095, -0.0476036

In [54]:
calibrate_and_plot_intervals(predictions_CatBoost_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

The empirical coverage before calibration is: 61.54%
The empirical coverage after calibration is: 97.44%


In [55]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_CatBoost_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

# **Quantile Regression with HistGradientBoosting**

In [56]:

# Assume best_params is a dictionary containing the best parameters for HistGradientBoostingRegressor
best_params = get_best_model_params(best_scores_autosampler, 'HGBR')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_HGBR_df = pd.DataFrame()

# Train a model for each quantile and make predictions using HistGradientBoostingRegressor
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'loss': 'quantile',
        'quantile': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = HistGradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_HGBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_HGBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_HGBR_df.head())

       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0  0.282605  0.268947  0.232755  0.279582  0.313398  0.349788  0.396859   
1  0.199734  0.237650  0.268777  0.276006  0.270776  0.296210  0.279390   
2  0.288116  0.767778  0.686172  0.696670  0.873640  0.919433  1.017245   
3  0.289965  0.442651  0.584939  0.539918  0.549212  0.539259  0.468894   
4  0.238497  0.291153  0.420557  0.239084  0.388019  0.460806  0.539422   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  0.378655  0.418861  0.448486  0.511176  0.856104  1.086456     0.4  
1  0.385591  0.440491  0.534318  0.462993  0.715266  0.924577     0.2  
2  1.017816  1.155153  1.192135  1.192340  1.165948  1.284874     1.1  
3  0.826827  0.809050  0.843803  0.878599  1.008869  1.093729     1.6  
4  0.491752  0.544439  0.639405  0.592708  0.848230  1.086456     0.2  


In [57]:
quantile_regression_print(predictions_HGBR_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/HGBM.xlsx")

First few actual values: [0.4 0.2 1.1 1.6 0.2]
Coverage of 90% prediction interval: 85.90%


In [58]:
evaluation_features_print(predictions_HGBR_df,x_test,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/HGBM.xlsx")

In [59]:
evaluate_uncertainity_matrix(predictions_HGBR_df, quantiles,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/HGBM.xlsx", model_name='HGBM')

Quantile crossing before sorting: 76 of 78 rows (97.44%)


{'empirical_quantiles': [np.float64(0.08974358974358974),
  np.float64(0.19230769230769232),
  np.float64(0.23076923076923078),
  np.float64(0.2948717948717949),
  np.float64(0.34615384615384615),
  np.float64(0.44871794871794873),
  np.float64(0.5128205128205128),
  np.float64(0.5512820512820513),
  np.float64(0.6410256410256411),
  np.float64(0.7435897435897436),
  np.float64(0.7692307692307693),
  np.float64(0.8717948717948718),
  np.float64(0.9487179487179487)],
 'average_interval_lengths': [np.float64(0.37743357614658113),
  np.float64(0.7519213202195221),
  np.float64(1.2085713487883665)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(0.526271687695701),
 'average_crps': np.float64(0.21075437268593966),
 'nll_values': array([-4.26651445e-01, -4.98329022e-01, -2.37808850e-01,  8.13971765e+00,
         2.03781798e-01,  2.94627950e+00,  9.44911187e-02, -3.39226526e-01,
        -2.96606800e-01,  7.40969545e-01, -5.41874619e-01, -3.85874233e-01,
         1.12272693e+

In [60]:
calibrate_and_plot_intervals(predictions_HGBR_df, "./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/HGBM.xlsx")

The empirical coverage before calibration is: 84.62%
The empirical coverage after calibration is: 76.92%


In [61]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_HGBR_df, "./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/HGBM.xlsx")

# **Quantile Regression with TabNet**

In [62]:
best_params = get_best_model_params(best_scores_autosampler, 'TabNet')

# Quantiles
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Storage
models_quantile = {}
predictions_TabNet_df = pd.DataFrame()

# Convert data to numpy (TabNet requirement)
X_train_np = np.array(X_train)
y_train_np = np.array(y_train).reshape(-1, 1)
X_test_np  = np.array(X_test)

# Quantile loss (pinball loss)
def quantile_loss(q):
    def loss(y_pred, y_true):
        error = y_true - y_pred
        return torch.mean(torch.maximum(q * error, (q - 1) * error))
    return loss

# FIX Q3: early stopping on a 20% validation split of TRAIN, scored with the pinball loss of
# the quantile being trained (was: eval_set = the training data itself, scored with MSE,
# so early stopping never triggered).
from pytorch_tabnet.metrics import Metric
X_fit_np, X_val_np, y_fit_np, y_val_np = train_test_split(X_train_np, y_train_np, test_size=0.2, random_state=42)

def pinball_metric(q):
    class Pinball(Metric):
        def __init__(self):
            self._name = f'pinball_{q}'
            self._maximize = False
        def __call__(self, y_true, y_score, weights=None):  # FIX B8: pytorch-tabnet2 also passes sample weights
            e = np.asarray(y_true, dtype=float) - np.asarray(y_score, dtype=float)
            return float(np.mean(np.maximum(q * e, (q - 1) * e)))
    return Pinball

# Train one TabNet per quantile
for q in quantiles:
    model = TabNetRegressor(
        **best_params,
    )

    model.fit(
        X_fit_np,
        y_fit_np,
        eval_set=[(X_val_np, y_val_np)],
        eval_metric=[pinball_metric(q)],
        loss_fn=quantile_loss(q),
        max_epochs=200,
        patience=30,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False
    )

    # Store model
    models_quantile[q] = model

    # Predict
    preds = model.predict(X_test_np).ravel()
    predictions_TabNet_df[q] = preds

# Add actual values
predictions_TabNet_df['Actual'] = y_test.values.ravel()

print(predictions_TabNet_df.head())


Early stopping occurred at epoch 46 with best_epoch = 16 and best_val_0_pinball_0.05 = 0.05177



Early stopping occurred at epoch 132 with best_epoch = 102 and best_val_0_pinball_0.1 = 0.09843



Early stopping occurred at epoch 142 with best_epoch = 112 and best_val_0_pinball_0.15 = 0.09404



Early stopping occurred at epoch 54 with best_epoch = 24 and best_val_0_pinball_0.2 = 0.16522



Early stopping occurred at epoch 103 with best_epoch = 73 and best_val_0_pinball_0.3 = 0.14415



Early stopping occurred at epoch 137 with best_epoch = 107 and best_val_0_pinball_0.4 = 0.1619


... (3 outputs left out)


Early stopping occurred at epoch 99 with best_epoch = 69 and best_val_0_pinball_0.8 = 0.11699



Early stopping occurred at epoch 76 with best_epoch = 46 and best_val_0_pinball_0.85 = 0.10384



Early stopping occurred at epoch 106 with best_epoch = 76 and best_val_0_pinball_0.9 = 0.07462



Early stopping occurred at epoch 100 with best_epoch = 70 and best_val_0_pinball_0.95 = 0.04371
       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0  0.062834  0.442876  0.362707  0.225716  0.458405  0.356754  0.401097   
1 -0.012623  0.063597  0.154490  0.522915  0.126365  0.039495  0.314607   
2  0.065219  0.393241  0.608046  0.638179  0.891303  0.780490  0.757375   
3  0.193581  0.557845  0.511406  0.593317  0.776826  0.559447  1.152017   
4  0.410419  0.429158  0.432604  0.290655  0.363885  0.332608  0.492769   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  0.375997  0.501242  0.322425  0.612672  0.373568  0.825801     0.4  
1  0.210682  0.174166  0.075301  0.497712  0.257879  1.312265     0.2  
2  1.166200  1.126987  1.004218  0.948185  1.180734  1.286563     1.1  
3  1.160329  1.128784  1.563378  1.095651  1.112060  1.886918     1.6  
4  0.447363  0.627036  0.667427  0.748317  0.717270  0.679853     0.2  


In [63]:
quantile_regression_print(predictions_TabNet_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/TabNet.xlsx")

First few actual values: [0.4 0.2 1.1 1.6 0.2]
Coverage of 90% prediction interval: 85.90%


In [64]:
evaluation_features_print(predictions_TabNet_df,x_test,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/TabNet.xlsx")

In [65]:
evaluate_uncertainity_matrix(predictions_TabNet_df, quantiles, "./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/TabNet.xlsx", model_name='TabNet')

Quantile crossing before sorting: 78 of 78 rows (100.0%)


{'empirical_quantiles': [np.float64(0.10256410256410256),
  np.float64(0.11538461538461539),
  np.float64(0.21794871794871795),
  np.float64(0.2948717948717949),
  np.float64(0.38461538461538464),
  np.float64(0.46153846153846156),
  np.float64(0.5128205128205128),
  np.float64(0.5897435897435898),
  np.float64(0.6282051282051282),
  np.float64(0.7307692307692307),
  np.float64(0.7692307692307693),
  np.float64(0.8717948717948718),
  np.float64(0.9615384615384616)],
 'average_interval_lengths': [np.float64(0.45150558803326046),
  np.float64(0.957695391602241),
  np.float64(1.5712993496503584)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(0.4173243080897851),
 'average_crps': np.float64(0.19599667298965726),
 'nll_values': array([-0.53704597,  0.01152545,  0.08608583,  0.7348625 ,  0.52726347,
         1.23478975,  0.42877117, -0.30288026, -0.69312585,  2.36917866,
        -0.40682557, -0.13982875,  1.39974124, -0.13989055, -0.43952447,
        -0.44345669, -0.135626

In [66]:
calibrate_and_plot_intervals(predictions_TabNet_df, "./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/TabNet.xlsx")

The empirical coverage before calibration is: 79.49%
The empirical coverage after calibration is: 82.05%


In [67]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_TabNet_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/TabNet.xlsx")

# **Quantile Regression with PGBM**

In [68]:
class PGBMWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, **params):
        self.params = params
        self.model = None

    def fit(self, X, y):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        y_ = y.to_numpy() if hasattr(y, "to_numpy") else np.array(y)
        self.model = PGBM()
        self.model.train(
            train_set=(X_, y_),
            objective=mseloss_objective,
            metric=rmseloss_metric,
            params=self.params
        )
        return self

    def predict(self, X):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        return self.model.predict(X_).numpy()


In [69]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# DataFrame to store predictions
predictions_PGBM_df = pd.DataFrame()

# Calculate and store quantiles
for q in quantiles:
    predictions_PGBM_df[q] = np.quantile(pred_dist, q, axis=0)

# Add actual target values to the DataFrame
predictions_PGBM_df['Actual'] = y_test.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_PGBM_df.head())

Training on CPU
Estimator 0/300, Train metric: 1.2173


Estimator 1/300, Train metric: 1.1687
Estimator 2/300, Train metric: 1.1274
Estimator 3/300, Train metric: 1.0832
Estimator 4/300, Train metric: 1.0460
Estimator 5/300, Train metric: 1.0061
Estimator 6/300, Train metric: 0.9684
Estimator 7/300, Train metric: 0.9329
Estimator 8/300, Train metric: 0.8993
Estimator 9/300, Train metric: 0.8678
Estimator 10/300, Train metric: 0.8406
Estimator 11/300, Train metric: 0.8139
Estimator 12/300, Train metric: 0.7866
Estimator 13/300, Train metric: 0.7607
Estimator 14/300, Train metric: 0.7364
Estimator 15/300, Train metric: 0.7125
Estimator 16/300, Train metric: 0.6900
Estimator 17/300, Train metric: 0.6688
Estimator 18/300, Train metric: 0.6486
Estimator 19/300, Train metric: 0.6299


Estimator 20/300, Train metric: 0.6117
Estimator 21/300, Train metric: 0.5945
Estimator 22/300, Train metric: 0.5781
Estimator 23/300, Train metric: 0.5627
Estimator 24/300, Train metric: 0.5481
Estimator 25/300, Train metric: 0.5346
Estimator 26/300, Train metric: 0.5216
Estimator 27/300, Train metric: 0.5096
Estimator 28/300, Train metric: 0.4972
Estimator 29/300, Train metric: 0.4855
Estimator 30/300, Train metric: 0.4740
Estimator 31/300, Train metric: 0.4631
Estimator 32/300, Train metric: 0.4516
Estimator 33/300, Train metric: 0.4414
Estimator 34/300, Train metric: 0.4316
Estimator 35/300, Train metric: 0.4224
Estimator 36/300, Train metric: 0.4135
Estimator 37/300, Train metric: 0.4051
Estimator 38/300, Train metric: 0.3965


Estimator 39/300, Train metric: 0.3891
Estimator 40/300, Train metric: 0.3821
Estimator 41/300, Train metric: 0.3748
Estimator 42/300, Train metric: 0.3688
Estimator 43/300, Train metric: 0.3631
Estimator 44/300, Train metric: 0.3576
Estimator 45/300, Train metric: 0.3519
Estimator 46/300, Train metric: 0.3466
Estimator 47/300, Train metric: 0.3418
Estimator 48/300, Train metric: 0.3373
Estimator 49/300, Train metric: 0.3324
Estimator 50/300, Train metric: 0.3286
Estimator 51/300, Train metric: 0.3249
Estimator 52/300, Train metric: 0.3215
Estimator 53/300, Train metric: 0.3182
Estimator 54/300, Train metric: 0.3139
Estimator 55/300, Train metric: 0.3109
Estimator 56/300, Train metric: 0.3082
Estimator 57/300, Train metric: 0.3057
Estimator 58/300, Train metric: 0.3017
Estimator 59/300, Train metric: 0.2995
Estimator 60/300, Train metric: 0.2972
Estimator 61/300, Train metric: 0.2953
Estimator 62/300, Train metric: 0.2931
Estimator 63/300, Train metric: 0.2911
Estimator 64/300, Train m

Estimator 72/300, Train metric: 0.2739
Estimator 73/300, Train metric: 0.2719
Estimator 74/300, Train metric: 0.2685
Estimator 75/300, Train metric: 0.2675
Estimator 76/300, Train metric: 0.2644
Estimator 77/300, Train metric: 0.2634
Estimator 78/300, Train metric: 0.2607
Estimator 79/300, Train metric: 0.2576
Estimator 80/300, Train metric: 0.2551
Estimator 81/300, Train metric: 0.2548
Estimator 82/300, Train metric: 0.2533
Estimator 83/300, Train metric: 0.2531
Estimator 84/300, Train metric: 0.2520
Estimator 85/300, Train metric: 0.2512
Estimator 86/300, Train metric: 0.2488
Estimator 87/300, Train metric: 0.2476
Estimator 88/300, Train metric: 0.2474
Estimator 89/300, Train metric: 0.2465
Estimator 90/300, Train metric: 0.2456
Estimator 91/300, Train metric: 0.2443
Estimator 92/300, Train metric: 0.2435
Estimator 93/300, Train metric: 0.2428
Estimator 94/300, Train metric: 0.2412
Estimator 95/300, Train metric: 0.2409
Estimator 96/300, Train metric: 0.2394
Estimator 97/300, Train m

Estimator 135/300, Train metric: 0.2393
Estimator 136/300, Train metric: 0.2393
Estimator 137/300, Train metric: 0.2393
Estimator 138/300, Train metric: 0.2393
Estimator 139/300, Train metric: 0.2393
Estimator 140/300, Train metric: 0.2393
Estimator 141/300, Train metric: 0.2393
Estimator 142/300, Train metric: 0.2393
Estimator 143/300, Train metric: 0.2393
Estimator 144/300, Train metric: 0.2393
Estimator 145/300, Train metric: 0.2393
Estimator 146/300, Train metric: 0.2393
Estimator 147/300, Train metric: 0.2393
Estimator 148/300, Train metric: 0.2393
Estimator 149/300, Train metric: 0.2393
Estimator 150/300, Train metric: 0.2393
Estimator 151/300, Train metric: 0.2393
Estimator 152/300, Train metric: 0.2393
Estimator 153/300, Train metric: 0.2393
Estimator 154/300, Train metric: 0.2393
Estimator 155/300, Train metric: 0.2393
Estimator 156/300, Train metric: 0.2393
Estimator 157/300, Train metric: 0.2393
Estimator 158/300, Train metric: 0.2393
Estimator 159/300, Train metric: 0.2393


       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0 -0.660923 -0.411019 -0.057119  0.031250  0.135315  0.274188  0.341554   
1 -0.614034 -0.322651 -0.269790 -0.006447  0.133035  0.277592  0.387668   
2 -0.016880  0.253683  0.510851  0.586418  0.813666  0.908789  0.995536   
3  0.137018  0.320186  0.437684  0.559198  0.667784  0.750463  0.838678   
4 -0.496977 -0.293617  0.043573  0.093584  0.291570  0.381616  0.515648   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  0.472242  0.595670  0.744184  0.888949  1.027598  1.375551     0.4  
1  0.496276  0.655190  0.915460  1.005439  1.166568  1.414633     0.2  
2  1.098232  1.202856  1.331882  1.458007  1.740235  2.063441     1.1  
3  0.904108  1.078036  1.345024  1.436642  1.474967  1.764087     1.6  
4  0.591708  0.727078  0.931764  1.022090  1.129131  1.327065     0.2  


In [70]:
quantile_regression_print(predictions_PGBM_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/PGBM.xlsx")

First few actual values: [0.4 0.2 1.1 1.6 0.2]
Coverage of 90% prediction interval: 98.72%


In [71]:
evaluation_features_print(predictions_PGBM_df,x_test,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/PGBM.xlsx")

In [72]:
evaluate_uncertainity_matrix(predictions_PGBM_df, quantiles, "./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/PGBM.xlsx", model_name='PGBM')

Quantile crossing before sorting: 0 of 78 rows (0.0%)


{'empirical_quantiles': [np.float64(0.0),
  np.float64(0.0),
  np.float64(0.01282051282051282),
  np.float64(0.038461538461538464),
  np.float64(0.2564102564102564),
  np.float64(0.41025641025641024),
  np.float64(0.5384615384615384),
  np.float64(0.6794871794871795),
  np.float64(0.7692307692307693),
  np.float64(0.8589743589743589),
  np.float64(0.9230769230769231),
  np.float64(0.9487179487179487),
  np.float64(0.9871794871794872)],
 'average_interval_lengths': [np.float64(0.7781316466892186),
  np.float64(1.3641091925539792),
  np.float64(1.9124169082333071)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(0.5117107685924723),
 'average_crps': np.float64(0.18824825421560748),
 'nll_values': array([0.44381707, 0.4818253 , 0.47430671, 1.39962399, 0.49123406,
        1.54646833, 0.42016866, 0.4572733 , 0.32120409, 0.68967462,
        0.44403831, 0.14878736, 0.32584353, 0.40791263, 0.49784577,
        0.54715678, 0.40327514, 0.31966582, 0.52770103, 0.35323483,
        

In [73]:
calibrate_and_plot_intervals(predictions_PGBM_df, "./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/PGBM.xlsx")

The empirical coverage before calibration is: 97.44%
The empirical coverage after calibration is: 92.31%


In [74]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_PGBM_df,"./drive/MyDrive/Regression/scour_uncertainty_analysis/quantile_regression/PGBM.xlsx")